# Blocking vs. Non-Blocking Synchronization

The ability for the sequencer to dispatch commands to other hardware subsystems but continue to run independently imposes the need for synchronization constructs. In previous examples involving synthesis and capture, the sequencer instructs the DMAs to begin streaming and then does nothing while waiting for them to complete. However, there are plenty of instances in which we want to trigger a DMA and then have the sequencer go off and do other things before the DMA is done, but then "resynchronize" afterwards by waiting for the DMA to complete. This is referred to as *non-blocking synchronization* because the sequencer's execution isn't "blocked" while other subsystems are doing something. 

The `RFDCSynchronizer` object that gets created when calling `Acadia.synchronizer()` can be configured for non-blocking synchronization. As a reminder, this object will keep track of all calls to `generate` and `capture` (as well as others) inside of its context and build up a list of all the `Channel` objects being used, but this is only necessary so that the `RFDCSynchronizer` can, upon exiting the context, add instructions that simultaneously trigger all of the appropriate DMAs and add a loop to block the sequencer until the DMAs have completed. However, if we want non-blocking synchronization, we need to instruct the `RFDCSynchronizer` to not add this latter loop. 

We'll demonstrate configuring the synchronizer for non-blocking synchronization by operating one of the ADCs as a "monitor" channel for a DAC. That is, we want to generate a series of pulses from some DAC but capture its output on an ADC the entire time. If we know ahead of time what pulses we want to generate and when, we could of course put everything inside of one blocking `Synchronizer`, but in situations where the sequencer needs to perform some action while pulses are still playing (such as queueing up additional pulses for continuous streaming), we need to exit the `Synchronizer` before the DMAs have stopped.

Let's start with a typical program setup:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

from acadia.system import Acadia
from acadia.channel import Channel

acadia = Acadia()

capture_time = 10e-6
pulse_time = 1e-6

pulse_channel = acadia.DAC(1)
capture_channel = acadia.ADC(1)

pulse_length = pulse_channel.seconds_to_samples(pulse_time)
capture_length = capture_channel.seconds_to_samples(capture_time)

pulse_memory = acadia.DACArray[pulse_channel.num](size=pulse_channel.seconds_to_bytes(pulse_time))
capture_memory = acadia.PLDDR0Array(size=capture_channel.seconds_to_bytes(capture_time))

# Create a sequence for the sequencer
@acadia.sequence
def sequence(a):

    # Create a non-blocking synchronization context
    # to begin capturing from an ADC "in the background"
    with a.synchronizer(block=False):
        a.capture(capture_channel, capture_memory)

    # Now we can go off and do other things
    # To demonstrate, we'll play two pulses back-to-back, 
    # block the sequencer for 100 cycles, then do both again
    for i in range(2):
        with a.synchronizer():
            a.generate(pulse_channel, pulse_memory)
            a.generate(pulse_channel, pulse_memory)

        with a.sequencer() as seq:
            for i in range(100):
                seq.nop()

# Load the wave memory with some square pulses
def program():   
    import numpy as np
    import time
    
    # Load the pulses into DAC memory
    pulse = np.ones(pulse_length, dtype=np.complex64)
    pulse_samples = pulse_channel.to_samples(pulse)
    acadia.memcpy(pulse_samples, pulse_memory)
    
    # Set up the channel properties
    pulse_channel.set_nyquist_zone(2)
    pulse_channel.configure_nco(frequency=1500e6)
    pulse_channel.set_vop(20000)
    capture_channel.set_nyquist_zone(2)
    capture_channel.set_dsa(0)
    
    # Clear the DDR array
    zeros = np.zeros(capture_length, dtype=np.complex64)
    zero_samples = capture_channel.to_samples(zeros)
    acadia.memcpy(zero_samples, capture_memory) 
    time.sleep(0.1) # Give the memory a moment to load
    
    # Configure the ADC switch
    acadia.configure()

    # Reset and run the sequencer
    acadia.sequencer_reset()
    acadia.sequencer_run(sequence)
    time.sleep(0.1)
    acadia.sequencer_halt()
    

Let's build and run the program like normal and plot the signal captured by the ADC:

In [ ]:
acadia.compile_all()
acadia.attach()
acadia.assemble(load=True)

program()

time_per_sample = capture_time / capture_channel.seconds_to_samples(capture_time)
trace = Channel.from_samples(capture_memory.memory)

times = np.arange(0, capture_time, time_per_sample)

fig,ax = plt.subplots()
ax.plot(times*1e6, np.real(trace), label="Re")
ax.plot(times*1e6, np.imag(trace), label="Im")
ax.set_xlabel("Time (us)")
ax.set_ylabel("Amplitude (\%FS)")
ax.grid()
ax.legend()

This output shows that the ADC begins capturing and the DAC plays two pulses. Then, the sequencer waits until only those pulses (and not the ADC capture) have completed, blocks for 100 cycles, and then repeats. Because we can capture both iterations of this loop, this means that the ADC's capture and the sequencer's sequence are running concurrently but independently.